In [7]:
# import data
import pandas as pd
URL = 'https://stepik.org/media/attachments/lesson/360344/bookings.csv'
bookings = pd.read_csv(URL, sep = ';')

bookings_head = bookings.head(7)

Hotel                         object
Is Canceled                    int64
Lead Time                      int64
arrival full date             object
Arrival Date Year              int64
Arrival Date Month            object
Arrival Date Week Number       int64
Arrival Date Day of Month      int64
Stays in Weekend nights        int64
Stays in week nights           int64
stays total nights             int64
Adults                         int64
Children                     float64
Babies                         int64
Meal                          object
Country                       object
Reserved Room Type            object
Assigned room type            object
customer type                 object
Reservation Status            object
Reservation status_date       object
dtype: object

In [8]:
bookings.shape


(119390, 21)

In [11]:
# working with the names of columns, 
# transform Is Cancelled -> is_cancelled

colmns = bookings.columns

new_columns = {}
for col in colmns:
    new_columns[col]  = col.replace(' ', '_').lower()

bookings = bookings.rename(columns = new_columns) 

bookings.head(5)

,hotel,is_canceled,lead_time,arrival_full_date,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,adults,children,babies,meal,country,reserved_room_type,assigned_room_type,customer_type,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015-07-01,2015,July,27,1,0,0,...,2,0.0,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015-07-01,2015,July,27,1,0,0,...,2,0.0,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015-07-01,2015,July,27,1,0,1,...,1,0.0,0,BB,GBR,A,C,Transient,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015-07-01,2015,July,27,1,0,1,...,1,0.0,0,BB,GBR,A,A,Transient,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015-07-01,2015,July,27,1,0,2,...,2,0.0,0,BB,GBR,A,A,Transient,Check-Out,2015-07-03


In [13]:
# Find top-5 countries with successful bookings (is_canceled = 0)

best_country = bookings.query('is_canceled == 0')['country']\
                       .value_counts()

In [15]:
best_country.head(5)

PRT    21071
GBR     9676
FRA     8481
ESP     6391
DEU     6069
Name: country, dtype: int64

In [59]:
# Find average number of nights in City Hotel and Resort Hotel

mean_bookings = bookings.groupby('hotel')\
                        .aggregate({'stays_total_nights': 'mean'}).round(2)

mean_bookings

,stays_total_nights
hotel,
City Hotel,2.98
Resort Hotel,4.32


In [21]:
# Find all casualties when assigned_room_type is distinct from reserved_room_type

bookings.query('reserved_room_type != assigned_room_type')['hotel'].count()

14917

In [25]:
# Find the most popular month in 2016
# Is it the same in 2017?

# group by year and month and take a sum

bookings.groupby(['arrival_date_year', 'arrival_date_month'])\
        .aggregate(total_bookings = ('hotel', 'count'))\
        .sort_values(['arrival_date_year', 'total_bookings'], ascending = False)

total_bookings
arrival_date_year arrival_date_month                
2017              May                           6313
                  April                         5661
                  June                          5647
                  July                          5313
                  March                         4970
                  August                        4925
                  February                      4177
                  January                       3681
2016              October                       6203
                  May                           5478
                  April                         5428
                  September                     5394
                  June                          5292
                  August                        5063
                  March                         4824
                  July                          4572
                  November                      4454
                  February                      3891
                  December                      3860
                  January                       2248
2015              September                     5114
                  October                       4957
                  August                        3889
                  December                      2920
                  July                          2776
                  November                      2340

In [34]:
# In which month (in a year) a booking in City Hotel is more likely being calcelled?

bookings.query("hotel == 'City Hotel'")\
        .groupby(['arrival_date_year','arrival_date_month'])\
        .aggregate(total_cancels =('is_canceled','sum'))\
        .sort_values(['arrival_date_year', 'total_cancels'], ascending = False)

total_cancels
arrival_date_year arrival_date_month               
2017              May                          2217
                  April                        1926
                  June                         1808
                  July                         1324
                  March                        1278
                  August                       1123
                  January                      1044
                  February                      971
2016              October                      1947
                  June                         1720
                  September                    1567
                  April                        1539
                  May                          1436
                  November                     1360
                  August                       1247
                  March                        1108
                  December                     1072
                  July                         1043
                  February                      930
                  January                       438
2015              September                    1543
                  October                      1321
                  August                       1232
                  July                          939
                  December                      668
                  November                      301

In [36]:
print(bookings['adults'].mean(),
bookings['children'].mean(),
bookings['babies'].mean())

1.8564033838679956 0.10388990333874994 0.007948739425412514


In [60]:
# let kids = children + babies and calculate the average number of kids for a hotel

bookings['kids'] = bookings['children'] + bookings['babies']

bookings.groupby('hotel').kids.mean().round(2)

hotel
City Hotel      0.10
Resort Hotel    0.14
Name: kids, dtype: float64

In [61]:
# calculate the Churn rate and check how it depends on having kids

bookings['has_kids'] = bookings['kids'] > 0 

bookings.head()

,hotel,is_canceled,lead_time,arrival_full_date,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,babies,meal,country,reserved_room_type,assigned_room_type,customer_type,reservation_status,reservation_status_date,kids,has_kids
0,Resort Hotel,0,342,2015-07-01,2015,July,27,1,0,0,...,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01,0.0,False
1,Resort Hotel,0,737,2015-07-01,2015,July,27,1,0,0,...,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01,0.0,False
2,Resort Hotel,0,7,2015-07-01,2015,July,27,1,0,1,...,0,BB,GBR,A,C,Transient,Check-Out,2015-07-02,0.0,False
3,Resort Hotel,0,13,2015-07-01,2015,July,27,1,0,1,...,0,BB,GBR,A,A,Transient,Check-Out,2015-07-02,0.0,False
4,Resort Hotel,0,14,2015-07-01,2015,July,27,1,0,2,...,0,BB,GBR,A,A,Transient,Check-Out,2015-07-03,0.0,False


In [50]:
 

agg_cancels = bookings.groupby('has_kids').agg(cancels = ('is_canceled','sum'), total_bookings = ('is_canceled', 'count'))
agg_cancels

,cancels,total_bookings
has_kids,,
False,40961,110054
True,3263,9336


In [58]:
# calculating the churn rate
round(agg_cancels['cancels']/agg_cancels['total_bookings']*100, 2)

has_kids
False    37.22
True     34.95
dtype: float64